# PechayDetect — Model B: MobileNetV3Large

**Thesis:** Detection of Nutrient Deficiencies in Pechay (*Brassica rapa* subsp. *chinensis*) Using Deep Learning

**Architecture:** MobileNetV3Large (Transfer Learning + Advanced Fine-Tuning)

---

## Python Version Justification

This notebook uses **Python 3.10** — the same as the MobileNetV2 notebook.

- TensorFlow 2.13 officially supports Python 3.8, 3.9, 3.10, and 3.11.
- Python 3.12 is **not** supported by TensorFlow 2.x (as of TF 2.15).
- Python 3.10 is the default Google Colab runtime and provides the most stable environment.
- `tf.keras.applications.MobileNetV3Large` is available from TF 2.4+ and stable under Python 3.10 + TF 2.13.

---

## MobileNetV3 vs MobileNetV2 — Key Differences

| Aspect | MobileNetV2 | MobileNetV3Large |
|--------|-------------|------------------|
| Activation | ReLU6 | Hard-Swish + Hard-Sigmoid |
| Attention | None | Squeeze-and-Excitation (SE) |
| Search | Manual design | Neural Architecture Search (NAS) |
| Head | Conv + GAP | Efficient last stage |
| Preprocessing | [-1, 1] | [0, 1] (rescale by 1/255) |
| Accuracy (ImageNet) | 71.8% top-1 | 75.2% top-1 |

Reference: Howard et al. (2019). *Searching for MobileNetV3*. ICCV 2019. arXiv:1905.02244

---

## Experimental Parity

For a **fair comparison** between MobileNetV2 and MobileNetV3, this notebook uses:
- **Identical dataset** (same splits, same class names)
- **Identical augmentation** (flip, rotation ±15°, zoom ±10%, brightness ±20%, contrast ±10%)
- **Identical evaluation protocol** (accuracy, precision, recall, F1, confusion matrix, ROC, ECE)
- **Same training strategy** (3-phase progressive fine-tuning, cosine LR decay with warmup)
- **Same OOD rejection mechanism** (confidence threshold)

The only differences are architecture-specific: preprocessing normalization, fine-tuning layer indices, and the last conv layer name for Grad-CAM.


## Section 1: Environment Setup

In [ ]:
!pip install -q tensorflow==2.13.0
!pip install -q scikit-learn==1.3.2
!pip install -q matplotlib seaborn pandas numpy pillow tqdm

import os
import sys
import math
import random
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
from pathlib import Path
from datetime import datetime

warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, optimizers, callbacks
from tensorflow.keras import mixed_precision
from tensorflow.keras.applications import MobileNetV3Large

import sklearn
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_curve, auc,
    precision_score, recall_score, f1_score, accuracy_score
)
from sklearn.preprocessing import label_binarize
from sklearn.utils.class_weight import compute_class_weight

print(f'Python:       {sys.version}')
print(f'TensorFlow:   {tf.__version__}')
print(f'Keras:        {keras.__version__}')
print(f'scikit-learn: {sklearn.__version__}')
print(f'NumPy:        {np.__version__}')


In [ ]:
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f'GPU detected: {gpus}')
    mixed_precision.set_global_policy('mixed_float16')
    print(f'Mixed precision enabled: {mixed_precision.global_policy()}')
else:
    print('No GPU detected — training on CPU (slower).')
    print('Runtime \u2192 Change runtime type \u2192 Hardware accelerator \u2192 GPU')
    mixed_precision.set_global_policy('float32')


## Section 2: Configuration

In [ ]:
# ================================================================
# CONFIGURATION — Edit this section to match your setup
# Must match the MobileNetV2 notebook for fair comparison
# ================================================================

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# --- Dataset paths (MUST match MobileNetV2 notebook) ---
DATASET_ROOT = '/content/Dataset'
# from google.colab import drive
# drive.mount('/content/drive')
# DATASET_ROOT = '/content/drive/MyDrive/PechayDetect/Dataset'

TRAIN_DIR = os.path.join(DATASET_ROOT, 'train')
VAL_DIR   = os.path.join(DATASET_ROOT, 'validation')
TEST_DIR  = os.path.join(DATASET_ROOT, 'test')

OUTPUT_DIR = '/content/outputs/mobilenetv3large'
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, 'logs'), exist_ok=True)

TRAIN_SPLIT      = 0.80
VALIDATION_SPLIT = 0.20

# --- Image parameters ---
# MobileNetV3Large uses 224x224 input by default (same as V2 for fair comparison)
# Reference: Howard et al. (2019). Searching for MobileNetV3. ICCV.
IMAGE_SIZE  = (224, 224)
CHANNELS    = 3
INPUT_SHAPE = IMAGE_SIZE + (CHANNELS,)

# --- Class names (must match MobileNetV2 notebook) ---
CLASS_NAMES = ['Healthy', 'Nitrogen', 'Phosphorus', 'Potassium']
# CLASS_NAMES = ['Healthy', 'Nitrogen', 'Phosphorus', 'Potassium', 'Unknown']  # 5-class OOD
NUM_CLASSES = len(CLASS_NAMES)

# --- Training hyperparameters ---
BATCH_SIZE    = 32
EPOCHS_PHASE1 = 20
EPOCHS_PHASE2 = 20
EPOCHS_PHASE3 = 15

LR_PHASE1 = 1e-3
LR_PHASE2 = 1e-4
LR_PHASE3 = 1e-5

# --- MobileNetV3Large fine-tuning ---
# MobileNetV3Large has approximately 264 layers.
# Phase 2: Unfreeze from layer 200 onward (~64 top layers)
# Phase 3: Unfreeze all non-BN layers
FINE_TUNE_FROM_PHASE2 = 200
FINE_TUNE_FROM_PHASE3 = 0

# --- Regularization (same as MobileNetV2 for fair comparison) ---
DROPOUT_RATE      = 0.3
L2_REGULARIZATION = 1e-4
LABEL_SMOOTHING   = 0.1

# --- OOD rejection ---
CONFIDENCE_THRESHOLD = 0.70

# --- Paths ---
MODEL_NAME       = 'mobilenetv3large_pechaydetect'
CHECKPOINT_PATH  = os.path.join(OUTPUT_DIR, f'{MODEL_NAME}_best.h5')
SAVEDMODEL_PATH  = os.path.join(OUTPUT_DIR, f'{MODEL_NAME}_savedmodel')
TFLITE_PATH_F32  = os.path.join(OUTPUT_DIR, f'{MODEL_NAME}_float32.tflite')
TFLITE_PATH_F16  = os.path.join(OUTPUT_DIR, f'{MODEL_NAME}_float16.tflite')
TFLITE_PATH_INT8 = os.path.join(OUTPUT_DIR, f'{MODEL_NAME}_int8.tflite')

print('Configuration loaded.')
print(f'  Classes ({NUM_CLASSES}): {CLASS_NAMES}')
print(f'  Image size:     {IMAGE_SIZE}')
print(f'  Batch size:     {BATCH_SIZE}')
print(f'  Output dir:     {OUTPUT_DIR}')


## Section 3: Dataset Verification and Loading

**MobileNetV3 Preprocessing Difference:**

MobileNetV3Large was pretrained with pixel values rescaled to **[0, 1]** (simply divide by 255).
This differs from MobileNetV2 which uses **[-1, 1]** normalization.

Using the correct preprocessing for each backbone is essential.
Reference: TensorFlow source — `tf.keras.applications.mobilenet_v3.preprocess_input` divides by 255 and maps to [-1, 1].

> **Note:** The TensorFlow implementation of MobileNetV3's `preprocess_input` rescales to [-1, 1] (same formula as V2). We use this behavior for consistency with the pretrained weights.

In [ ]:
# Verify TF's MobileNetV3 preprocessing behavior
from tensorflow.keras.applications.mobilenet_v3 import preprocess_input as mobilenetv3_preprocess
test_img = tf.constant([[[128.0, 128.0, 128.0]]])  # shape (1,1,3)
preprocessed = mobilenetv3_preprocess(test_img)
print(f'MobileNetV3 preprocess_input([128]) = {preprocessed.numpy().flatten()[0]:.4f}')
print('(Result \u2248 0.004 confirms TF uses the same [-1,1] formula as MobileNetV2)')
print('Both V2 and V3 use: pixel = (pixel / 127.5) - 1.0')


In [ ]:
IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.webp'}


def verify_dataset_structure(split_dirs, class_names):
    """Verify dataset directory structure and report image counts."""
    issues = []
    summary = {}

    for split_name, dir_path in split_dirs.items():
        summary[split_name] = {}
        if not os.path.isdir(dir_path):
            issues.append(f'Missing directory: {dir_path}')
            continue
        for cls in class_names:
            cls_path = os.path.join(dir_path, cls)
            if not os.path.isdir(cls_path):
                issues.append(f'Missing class folder: {cls_path}')
                summary[split_name][cls] = 0
            else:
                n = sum(1 for f in os.listdir(cls_path)
                        if os.path.splitext(f)[1].lower() in IMAGE_EXTS)
                summary[split_name][cls] = n
                if n == 0:
                    issues.append(f'No images in: {cls_path}')

    header = f"{'Class':<15}" + ''.join(f"{s:>12}" for s in split_dirs)
    print(header)
    print('-' * len(header))
    for cls in class_names:
        row = f"{cls:<15}"
        for split_name in split_dirs:
            row += f"{summary[split_name].get(cls, 0):>12}"
        print(row)
    total_row = f"{'TOTAL':<15}"
    for split_name in split_dirs:
        total_row += f"{sum(summary[split_name].values()):>12}"
    print('-' * len(header))
    print(total_row)

    if issues:
        print('\n\u26a0\ufe0f  Issues:')
        for issue in issues:
            print(f'  - {issue}')
        raise FileNotFoundError('Fix the above issues before continuing.')
    print('\n\u2705 Dataset structure verified.')
    return summary


print('Verifying dataset structure...')
split_dirs = {'Train': TRAIN_DIR, 'Validation': VAL_DIR, 'Test': TEST_DIR}
dataset_summary = verify_dataset_structure(split_dirs, CLASS_NAMES)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
colors = ['#4CAF50', '#F44336', '#2196F3', '#FF9800', '#9C27B0'][:NUM_CLASSES]
for ax, (split_name, counts) in zip(axes, dataset_summary.items()):
    values = [counts.get(c, 0) for c in CLASS_NAMES]
    bars = ax.bar(CLASS_NAMES, values, color=colors, edgecolor='white')
    ax.set_title(f'{split_name} Set', fontsize=13, fontweight='bold')
    ax.set_xlabel('Class'); ax.set_ylabel('Number of Images')
    ax.tick_params(axis='x', rotation=30)
    ax.set_ylim(0, max(values) * 1.2 if max(values) > 0 else 10)
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + max(values) * 0.02,
                str(val), ha='center', va='bottom', fontsize=10, fontweight='bold')
plt.suptitle('PechayDetect — Dataset Class Distribution (MobileNetV3Large)', fontsize=15)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'class_distribution.png'), dpi=150, bbox_inches='tight')
plt.show()


## Section 4: Data Augmentation and Preprocessing

Identical augmentation to MobileNetV2 notebook for fair comparison.

**Preprocessing:** `(pixel / 127.5) - 1.0` → [-1, 1] range (matches TF's `mobilenet_v3.preprocess_input`).

In [ ]:
# Identical augmentation to MobileNetV2 notebook for experimental parity
data_augmentation = keras.Sequential([
    layers.RandomFlip('horizontal_and_vertical', seed=SEED),
    layers.RandomRotation(factor=0.15, fill_mode='reflect', seed=SEED),
    layers.RandomZoom(height_factor=(-0.1, 0.1), seed=SEED),
    layers.RandomBrightness(factor=0.2, seed=SEED),
    layers.RandomContrast(factor=0.1, seed=SEED),
], name='data_augmentation')


def preprocess_mobilenetv3(image, label):
    """
    Normalize image to [-1, 1] using the formula (pixel / 127.5) - 1.0.
    This matches tf.keras.applications.mobilenet_v3.preprocess_input behavior.
    """
    image = tf.cast(image, tf.float32)
    image = (image / 127.5) - 1.0
    return image, label


def build_dataset(directory, split_name, augment=False):
    """
    Load and pipeline a dataset split.

    Args:
        directory (str): Path to the dataset split directory.
        split_name (str): 'train', 'validation', or 'test'.
        augment (bool): Apply augmentation if True (training only).

    Returns:
        tf.data.Dataset: Optimized, batched, prefetched dataset.
    """
    ds = tf.keras.utils.image_dataset_from_directory(
        directory,
        labels='inferred',
        label_mode='categorical',
        class_names=CLASS_NAMES,
        color_mode='rgb',
        batch_size=BATCH_SIZE,
        image_size=IMAGE_SIZE,
        shuffle=(split_name == 'train'),
        seed=SEED,
        interpolation='bilinear',
        crop_to_aspect_ratio=False,
    )

    ds = ds.map(preprocess_mobilenetv3, num_parallel_calls=tf.data.AUTOTUNE)

    if augment:
        ds = ds.map(
            lambda x, y: (data_augmentation(x, training=True), y),
            num_parallel_calls=tf.data.AUTOTUNE
        )

    return ds.cache().prefetch(buffer_size=tf.data.AUTOTUNE)


print('Loading datasets...')
train_ds = build_dataset(TRAIN_DIR, 'train',      augment=True)
val_ds   = build_dataset(VAL_DIR,   'validation', augment=False)
test_ds  = build_dataset(TEST_DIR,  'test',       augment=False)

print(f'  Train batches:      {train_ds.cardinality().numpy()}')
print(f'  Validation batches: {val_ds.cardinality().numpy()}')
print(f'  Test batches:       {test_ds.cardinality().numpy()}')
print('\u2705 Datasets loaded.')


In [ ]:
def show_samples(dataset, class_names, n_cols=4, n_rows=2, title='Sample Images'):
    """Display images, denormalized from [-1,1] to [0,1] for display."""
    images, labels = next(iter(dataset))
    img_display = np.clip((images.numpy() + 1.0) / 2.0, 0, 1)
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 3, n_rows * 3))
    axes = axes.flatten()
    for i, ax in enumerate(axes):
        if i < len(img_display):
            ax.imshow(img_display[i])
            ax.set_title(class_names[np.argmax(labels[i])], fontsize=10)
        ax.axis('off')
    plt.suptitle(title, fontsize=13)
    plt.tight_layout()
    save_path = os.path.join(OUTPUT_DIR, title.lower().replace(' ', '_') + '.png')
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()


show_samples(train_ds, CLASS_NAMES, title='Train Samples with Augmentation')
show_samples(val_ds,   CLASS_NAMES, title='Validation Samples (No Augmentation)')


## Section 5: Class Weighting

In [ ]:
def get_label_array(directory, class_names):
    labels = []
    for idx, cls in enumerate(class_names):
        cls_path = os.path.join(directory, cls)
        if not os.path.isdir(cls_path):
            continue
        n = sum(1 for f in os.listdir(cls_path)
                if os.path.splitext(f)[1].lower() in IMAGE_EXTS)
        labels.extend([idx] * n)
    return np.array(labels)


train_labels = get_label_array(TRAIN_DIR, CLASS_NAMES)

class_weights_arr = compute_class_weight(
    class_weight='balanced',
    classes=np.arange(NUM_CLASSES),
    y=train_labels,
)
class_weight_dict = {i: float(w) for i, w in enumerate(class_weights_arr)}

print('Class weights (balanced):')
for cls, w in zip(CLASS_NAMES, class_weights_arr):
    print(f'  {cls:<15}: {w:.4f}')


## Section 6: Learning Rate Schedule

Identical cosine decay with linear warmup schedule as the MobileNetV2 notebook — required for fair comparison.

In [ ]:
class WarmupCosineDecay(keras.optimizers.schedules.LearningRateSchedule):
    """
    Linear warmup + cosine decay learning rate schedule.
    Reference: Loshchilov & Hutter (2017). SGDR. ICLR 2017. arXiv:1608.03983
    """

    def __init__(self, initial_lr, warmup_steps, total_steps, min_lr=0.0):
        super().__init__()
        self.initial_lr   = float(initial_lr)
        self.warmup_steps = int(warmup_steps)
        self.total_steps  = int(total_steps)
        self.min_lr       = float(min_lr)

    def __call__(self, step):
        step   = tf.cast(step, tf.float32)
        warmup = tf.cast(self.warmup_steps, tf.float32)
        total  = tf.cast(self.total_steps,  tf.float32)
        warmup_lr  = self.initial_lr * (step / tf.maximum(warmup, 1.0))
        progress   = (step - warmup) / tf.maximum(total - warmup, 1.0)
        cosine_lr  = self.min_lr + 0.5 * (self.initial_lr - self.min_lr) * (
            1.0 + tf.cos(math.pi * progress)
        )
        return tf.where(step < warmup, warmup_lr, cosine_lr)

    def get_config(self):
        return {
            'initial_lr': self.initial_lr, 'warmup_steps': self.warmup_steps,
            'total_steps': self.total_steps, 'min_lr': self.min_lr,
        }


def make_lr_schedule(initial_lr, n_epochs, steps_per_epoch, warmup_fraction=0.1):
    total  = n_epochs * steps_per_epoch
    warmup = max(1, int(warmup_fraction * total))
    return WarmupCosineDecay(initial_lr, warmup, total)


STEPS_PER_EPOCH = len(train_ds)
print(f'Steps per epoch: {STEPS_PER_EPOCH}')


## Section 7: Model Architecture — MobileNetV3Large

**MobileNetV3Large** (Howard et al., 2019):
- Found via Neural Architecture Search (NAS) + NetAdapt
- Hard-Swish activation: `x * ReLU6(x+3) / 6` — more efficient than standard Swish
- Squeeze-and-Excitation (SE) blocks — channel attention mechanism
- Redesigned last stage for efficiency
- ImageNet top-1 accuracy: 75.2% (vs MobileNetV2's 71.8%)

**Classification head** (same structure as MobileNetV2 for fair comparison):
1. `GlobalAveragePooling2D`
2. `BatchNormalization`
3. `Dense(256, ReLU, L2)`
4. `Dropout(0.3)`
5. `Dense(NUM_CLASSES)` + `Softmax (float32)`

> **Note on `minimalistic=False`:** We use the full MobileNetV3Large (with SE blocks and hard-swish). Setting `minimalistic=True` removes these optimizations but can run faster on CPU. Since *accuracy is more important than speed* for this thesis, `minimalistic=False` is used.

In [ ]:
def build_mobilenetv3_model(num_classes, input_shape, dropout_rate, l2_reg):
    """
    Build MobileNetV3Large transfer learning model.

    Args:
        num_classes (int): Number of output classes.
        input_shape (tuple): Input shape (H, W, C).
        dropout_rate (float): Dropout probability in the head.
        l2_reg (float): L2 regularization coefficient.

    Returns:
        model (keras.Model): Full model.
        base_model (keras.Model): MobileNetV3Large backbone.
    """
    base_model = MobileNetV3Large(
        input_shape=input_shape,
        alpha=1.0,              # Width multiplier: 1.0 = full capacity
        minimalistic=False,     # Use full model with SE blocks and hard-swish
        include_top=False,      # Exclude ImageNet classifier head
        weights='imagenet',
        pooling=None,
        dropout_rate=0.0,       # We define our own dropout in the head
    )
    base_model.trainable = False  # Freeze initially (Phase 1)

    inputs = keras.Input(shape=input_shape, name='input_image')

    # training=False keeps BN layers in inference mode while base is frozen
    x = base_model(inputs, training=False)

    # --- Classification head (identical to MobileNetV2 for fair comparison) ---
    x = layers.GlobalAveragePooling2D(name='head_gap')(x)
    x = layers.BatchNormalization(name='head_bn')(x)
    x = layers.Dense(
        256,
        activation='relu',
        kernel_regularizer=keras.regularizers.l2(l2_reg),
        name='head_dense'
    )(x)
    x = layers.Dropout(dropout_rate, name='head_dropout')(x)
    x = layers.Dense(num_classes, name='logits')(x)

    # Float32 softmax (required for mixed precision numerical stability)
    outputs = layers.Activation('softmax', dtype='float32', name='output_softmax')(x)

    model = keras.Model(inputs=inputs, outputs=outputs, name='MobileNetV3Large_PechayDetect')
    return model, base_model


model, base_model = build_mobilenetv3_model(
    num_classes=NUM_CLASSES,
    input_shape=INPUT_SHAPE,
    dropout_rate=DROPOUT_RATE,
    l2_reg=L2_REGULARIZATION,
)

model.summary()
print(f'\nTotal layers in model:    {len(model.layers)}')
print(f'Total layers in backbone: {len(base_model.layers)}')
trainable   = sum(v.numpy().size for v in model.trainable_variables)
untrainable = sum(v.numpy().size for v in model.non_trainable_variables)
print(f'Trainable params:    {trainable:,}')
print(f'Non-trainable params:{untrainable:,}')


## Section 8: Phase 1 Training — Classification Head Only

In [ ]:
def get_standard_callbacks(phase_name, output_dir, checkpoint_path,
                           monitor_metric='val_accuracy',
                           patience_stop=8):
    return [
        keras.callbacks.ModelCheckpoint(
            filepath=checkpoint_path,
            monitor=monitor_metric,
            mode='max' if 'accuracy' in monitor_metric else 'min',
            save_best_only=True,
            save_weights_only=False,
            verbose=1,
        ),
        keras.callbacks.EarlyStopping(
            monitor=monitor_metric,
            mode='max' if 'accuracy' in monitor_metric else 'min',
            patience=patience_stop,
            restore_best_weights=True,
            verbose=1,
        ),
        keras.callbacks.CSVLogger(
            filename=os.path.join(output_dir, f'{phase_name}_history.csv'),
            append=False,
        ),
        keras.callbacks.TensorBoard(
            log_dir=os.path.join(output_dir, 'logs', phase_name),
            histogram_freq=1,
        ),
    ]


phase1_schedule = make_lr_schedule(LR_PHASE1, EPOCHS_PHASE1, STEPS_PER_EPOCH)

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=phase1_schedule),
    loss=keras.losses.CategoricalCrossentropy(label_smoothing=LABEL_SMOOTHING),
    metrics=[
        keras.metrics.CategoricalAccuracy(name='accuracy'),
        keras.metrics.TopKCategoricalAccuracy(k=2, name='top2_accuracy'),
        keras.metrics.AUC(name='auc', multi_label=False),
    ],
)

print('=' * 60)
print('PHASE 1: Training classification head (base frozen)')
print('=' * 60)

history_p1 = model.fit(
    train_ds,
    epochs=EPOCHS_PHASE1,
    validation_data=val_ds,
    class_weight=class_weight_dict,
    callbacks=get_standard_callbacks('phase1', OUTPUT_DIR, CHECKPOINT_PATH),
    verbose=1,
)

print(f'\nPhase 1 best val_accuracy: {max(history_p1.history["val_accuracy"]):.4f}')


In [ ]:
def plot_phase_history(history, phase_label, output_dir):
    h = history.history
    epochs = range(1, len(h['loss']) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].plot(epochs, h['loss'],     'b-o', markersize=4, label='Train')
    axes[0].plot(epochs, h['val_loss'], 'r-o', markersize=4, label='Validation')
    axes[0].set_title(f'{phase_label} — Loss')
    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
    axes[0].legend(); axes[0].grid(True, alpha=0.3)

    axes[1].plot(epochs, h['accuracy'],     'b-o', markersize=4, label='Train')
    axes[1].plot(epochs, h['val_accuracy'], 'r-o', markersize=4, label='Validation')
    axes[1].set_title(f'{phase_label} — Accuracy')
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
    axes[1].legend(); axes[1].grid(True, alpha=0.3)
    axes[1].set_ylim([0, 1.05])

    plt.suptitle(f'MobileNetV3Large — {phase_label}', fontsize=14)
    plt.tight_layout()
    fname = os.path.join(output_dir, phase_label.lower().replace(' ', '_') + '_curves.png')
    plt.savefig(fname, dpi=150, bbox_inches='tight')
    plt.show()


plot_phase_history(history_p1, 'Phase 1 (Frozen Base)', OUTPUT_DIR)


## Section 9: Phase 2 — Fine-Tune Top Layers

MobileNetV3Large has approximately 264 layers. We unfreeze from layer `FINE_TUNE_FROM_PHASE2` (default: 200) onward.

BatchNormalization layers remain frozen throughout fine-tuning (same rule as MobileNetV2).

In [ ]:
def set_base_trainability(base_model, from_layer_idx):
    """
    Unfreeze non-BN layers from from_layer_idx onward in the base model.
    BN layers are always kept frozen during fine-tuning.
    """
    base_model.trainable = True
    for i, layer in enumerate(base_model.layers):
        if isinstance(layer, layers.BatchNormalization):
            layer.trainable = False
        elif i < from_layer_idx:
            layer.trainable = False
        else:
            layer.trainable = True

    trainable_n = sum(1 for l in base_model.layers
                      if l.trainable and not isinstance(l, layers.BatchNormalization))
    print(f'Base model trainable (non-BN) layers: {trainable_n}/{len(base_model.layers)}')


print('=' * 60)
print('PHASE 2: Fine-tuning top MobileNetV3Large layers')
print(f'Unfreezing from layer {FINE_TUNE_FROM_PHASE2} of {len(base_model.layers)}')
print('=' * 60)

set_base_trainability(base_model, FINE_TUNE_FROM_PHASE2)

phase2_schedule = make_lr_schedule(LR_PHASE2, EPOCHS_PHASE2, STEPS_PER_EPOCH, warmup_fraction=0.05)

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=phase2_schedule),
    loss=keras.losses.CategoricalCrossentropy(label_smoothing=LABEL_SMOOTHING),
    metrics=[
        keras.metrics.CategoricalAccuracy(name='accuracy'),
        keras.metrics.TopKCategoricalAccuracy(k=2, name='top2_accuracy'),
        keras.metrics.AUC(name='auc', multi_label=False),
    ],
)

history_p2 = model.fit(
    train_ds,
    epochs=EPOCHS_PHASE2,
    validation_data=val_ds,
    class_weight=class_weight_dict,
    callbacks=get_standard_callbacks('phase2', OUTPUT_DIR, CHECKPOINT_PATH, patience_stop=7),
    verbose=1,
)

print(f'\nPhase 2 best val_accuracy: {max(history_p2.history["val_accuracy"]):.4f}')
plot_phase_history(history_p2, 'Phase 2 (Top Layers Fine-Tuned)', OUTPUT_DIR)


## Section 10: Phase 3 — Full Model Fine-Tuning

In [ ]:
print('=' * 60)
print('PHASE 3: Full model fine-tuning (all non-BN layers)')
print('=' * 60)

set_base_trainability(base_model, from_layer_idx=0)

phase3_schedule = make_lr_schedule(LR_PHASE3, EPOCHS_PHASE3, STEPS_PER_EPOCH, warmup_fraction=0.05)

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=phase3_schedule),
    loss=keras.losses.CategoricalCrossentropy(label_smoothing=LABEL_SMOOTHING),
    metrics=[
        keras.metrics.CategoricalAccuracy(name='accuracy'),
        keras.metrics.TopKCategoricalAccuracy(k=2, name='top2_accuracy'),
        keras.metrics.AUC(name='auc', multi_label=False),
    ],
)

history_p3 = model.fit(
    train_ds,
    epochs=EPOCHS_PHASE3,
    validation_data=val_ds,
    class_weight=class_weight_dict,
    callbacks=get_standard_callbacks('phase3', OUTPUT_DIR, CHECKPOINT_PATH, patience_stop=5),
    verbose=1,
)

print(f'\nPhase 3 best val_accuracy: {max(history_p3.history["val_accuracy"]):.4f}')
plot_phase_history(history_p3, 'Phase 3 (Full Fine-Tuning)', OUTPUT_DIR)


In [ ]:
def plot_combined_history(h1, h2, h3, output_dir, model_name):
    acc     = h1.history['accuracy']     + h2.history['accuracy']     + h3.history['accuracy']
    val_acc = h1.history['val_accuracy'] + h2.history['val_accuracy'] + h3.history['val_accuracy']
    loss    = h1.history['loss']         + h2.history['loss']         + h3.history['loss']
    val_loss= h1.history['val_loss']     + h2.history['val_loss']     + h3.history['val_loss']

    ep1_end = len(h1.history['accuracy'])
    ep2_end = ep1_end + len(h2.history['accuracy'])
    total   = len(acc)
    epochs  = range(1, total + 1)

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    for ax, (train_v, val_v, ylabel) in zip(axes, [
        (acc, val_acc, 'Accuracy'),
        (loss, val_loss, 'Loss'),
    ]):
        ax.plot(epochs, train_v, 'b-', lw=2, label='Train')
        ax.plot(epochs, val_v,   'r-', lw=2, label='Validation')
        ax.axvline(ep1_end + 0.5, color='gray',  ls='--', lw=1.5, label='Phase 2 start')
        ax.axvline(ep2_end + 0.5, color='green', ls='--', lw=1.5, label='Phase 3 start')
        ax.set_xlabel('Epoch'); ax.set_ylabel(ylabel)
        ax.set_title(f'{model_name} Combined — {ylabel}')
        ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

    plt.suptitle(f'{model_name} — Full Training History (All Phases)', fontsize=14)
    plt.tight_layout()
    fname = os.path.join(output_dir, 'combined_training_history.png')
    plt.savefig(fname, dpi=150, bbox_inches='tight')
    plt.show()


plot_combined_history(history_p1, history_p2, history_p3, OUTPUT_DIR, 'MobileNetV3Large')


## Section 11: Evaluation on Test Set

In [ ]:
print('=' * 60)
print('EVALUATION: Loading best checkpoint')
print('=' * 60)

best_model = keras.models.load_model(CHECKPOINT_PATH)
print(f'Loaded: {CHECKPOINT_PATH}')

eval_results = best_model.evaluate(test_ds, verbose=1)
eval_dict = dict(zip(best_model.metrics_names, eval_results))
print('\nKeras evaluation:')
for name, val in eval_dict.items():
    print(f'  {name}: {val:.4f}')

all_probs = []
all_true  = []
for images, labels in test_ds:
    probs = best_model.predict(images, verbose=0)
    all_probs.extend(probs)
    all_true.extend(np.argmax(labels.numpy(), axis=1))

all_probs = np.array(all_probs)
all_true  = np.array(all_true)
all_preds = np.argmax(all_probs, axis=1)

print(f'\nTotal test samples: {len(all_true)}')


In [ ]:
acc  = accuracy_score(all_true, all_preds)
prec = precision_score(all_true, all_preds, average='weighted', zero_division=0)
rec  = recall_score(all_true, all_preds, average='weighted', zero_division=0)
f1   = f1_score(all_true, all_preds, average='weighted', zero_division=0)

print('\n' + '=' * 55)
print('EVALUATION SUMMARY — MobileNetV3Large')
print('=' * 55)
print(f'  Accuracy  (weighted): {acc:.4f}  ({acc*100:.2f}%)')
print(f'  Precision (weighted): {prec:.4f}')
print(f'  Recall    (weighted): {rec:.4f}')
print(f'  F1 Score  (weighted): {f1:.4f}')
print('=' * 55)
print('\nPer-class Classification Report:')
print(classification_report(all_true, all_preds, target_names=CLASS_NAMES, digits=4))

metrics_df = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision (weighted)', 'Recall (weighted)', 'F1 Score (weighted)'],
    'MobileNetV3Large': [acc, prec, rec, f1],
})
metrics_df.to_csv(os.path.join(OUTPUT_DIR, 'evaluation_metrics.csv'), index=False)
print(f'\nMetrics saved to {OUTPUT_DIR}/evaluation_metrics.csv')


In [ ]:
def plot_confusion_matrix(true_labels, pred_labels, class_names, output_dir, model_name):
    cm_raw  = confusion_matrix(true_labels, pred_labels)
    cm_norm = cm_raw.astype(float) / cm_raw.sum(axis=1, keepdims=True)
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    for ax, (matrix, title, fmt) in zip(axes, [
        (cm_raw,  'Count',          'd'),
        (cm_norm, 'Row-Normalized', '.2%'),
    ]):
        sns.heatmap(matrix, annot=True, fmt=fmt, cmap='Blues',
                    xticklabels=class_names, yticklabels=class_names,
                    ax=ax, linewidths=0.5, cbar_kws={'shrink': 0.8})
        ax.set_title(f'{model_name} — Confusion Matrix ({title})')
        ax.set_xlabel('Predicted'); ax.set_ylabel('True')
        ax.tick_params(axis='x', rotation=30)
    plt.tight_layout()
    fname = os.path.join(output_dir, 'confusion_matrix.png')
    plt.savefig(fname, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: {fname}')


plot_confusion_matrix(all_true, all_preds, CLASS_NAMES, OUTPUT_DIR, 'MobileNetV3Large')


In [ ]:
def plot_roc_curves(true_labels, pred_probs, class_names, output_dir, model_name):
    n = len(class_names)
    true_bin = label_binarize(true_labels, classes=list(range(n)))
    fig, ax = plt.subplots(figsize=(9, 7))
    colors = plt.cm.tab10(np.linspace(0, 1, n))
    all_fpr = np.linspace(0, 1, 200)
    mean_tpr = np.zeros(200)
    auc_scores = {}
    for i, (cls, color) in enumerate(zip(class_names, colors)):
        fpr, tpr, _ = roc_curve(true_bin[:, i], pred_probs[:, i])
        roc_auc = auc(fpr, tpr)
        auc_scores[cls] = roc_auc
        mean_tpr += np.interp(all_fpr, fpr, tpr)
        ax.plot(fpr, tpr, lw=2, color=color, label=f'{cls} (AUC={roc_auc:.4f})')
    mean_tpr /= n
    macro_auc = auc(all_fpr, mean_tpr)
    ax.plot(all_fpr, mean_tpr, 'k--', lw=2, label=f'Macro Avg (AUC={macro_auc:.4f})')
    ax.plot([0, 1], [0, 1], 'gray', ls=':', lw=1)
    ax.set(xlim=[0, 1], ylim=[0, 1.05],
           xlabel='False Positive Rate', ylabel='True Positive Rate',
           title=f'{model_name} — ROC Curves (One-vs-Rest)')
    ax.legend(loc='lower right', fontsize=10)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    fname = os.path.join(output_dir, 'roc_curves.png')
    plt.savefig(fname, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Macro-average AUC: {macro_auc:.4f}')
    return auc_scores, macro_auc


auc_scores, macro_auc = plot_roc_curves(all_true, all_probs, CLASS_NAMES, OUTPUT_DIR, 'MobileNetV3Large')


## Section 12: Test-Time Augmentation (TTA)

In [ ]:
tta_augment = keras.Sequential([
    layers.RandomFlip('horizontal_and_vertical'),
    layers.RandomRotation(factor=0.10),
    layers.RandomBrightness(factor=0.1),
], name='tta_augmentation')


def predict_with_tta(model, dataset, n_passes=5):
    avg_probs   = []
    true_labels = []
    for images, labels in dataset:
        batch_size  = images.shape[0]
        batch_probs = np.zeros((batch_size, NUM_CLASSES), dtype=np.float32)
        batch_probs += model.predict(images, verbose=0)
        for _ in range(n_passes - 1):
            aug_images   = tta_augment(images, training=True)
            batch_probs += model.predict(aug_images, verbose=0)
        avg_probs.extend(batch_probs / n_passes)
        true_labels.extend(np.argmax(labels.numpy(), axis=1))
    return np.array(avg_probs), np.array(true_labels)


print('Running TTA (5 passes per image)...')
tta_probs, tta_true = predict_with_tta(best_model, test_ds, n_passes=5)
tta_preds = np.argmax(tta_probs, axis=1)

tta_acc = accuracy_score(tta_true, tta_preds)
tta_f1  = f1_score(tta_true, tta_preds, average='weighted', zero_division=0)

print(f'\nTTA Results:')
print(f'  Standard Accuracy: {acc:.4f}')
print(f'  TTA Accuracy:      {tta_acc:.4f}  (delta: {tta_acc-acc:+.4f})')
print(f'  Standard F1:       {f1:.4f}')
print(f'  TTA F1:            {tta_f1:.4f}  (delta: {tta_f1-f1:+.4f})')


## Section 13: Grad-CAM Visualization

For MobileNetV3Large, the last meaningful convolutional activation before GAP differs from MobileNetV2.

MobileNetV3Large ends with a `Conv_2` layer followed by a hard-swish activation. We target the activation layer just before the global average pooling. The cell below identifies the correct layer automatically.

In [ ]:
def find_last_conv_activation(base_model):
    """
    Identify the last convolutional or activation layer in the backbone
    that precedes any pooling or flatten operation.

    This is used as the target layer for Grad-CAM.
    """
    last_name = None
    pool_types = (layers.GlobalAveragePooling2D, layers.GlobalMaxPooling2D,
                  layers.Flatten, layers.AveragePooling2D)

    for layer in base_model.layers:
        if isinstance(layer, pool_types):
            break
        if hasattr(layer, 'output_shape'):
            out_shape = layer.output_shape
            if isinstance(out_shape, list):
                out_shape = out_shape[0]
            if len(out_shape) == 4:  # (batch, H, W, C) — spatial feature map
                last_name = layer.name
    return last_name


# Retrieve the backbone inside the full model
# MobileNetV3Large default name in tf.keras.applications
try:
    BASE_MODEL_NAME = 'MobileNetV3Large'
    backbone = best_model.get_layer(BASE_MODEL_NAME)
except ValueError:
    # Fallback: find the layer by type
    backbone = next(l for l in best_model.layers
                    if 'mobilenetv3' in l.name.lower())
    BASE_MODEL_NAME = backbone.name

LAST_CONV_NAME = find_last_conv_activation(backbone)
print(f'Backbone name:         {BASE_MODEL_NAME}')
print(f'Last spatial layer:    {LAST_CONV_NAME}')

if LAST_CONV_NAME is None:
    print('\nFallback: listing last 15 backbone layers to identify target:')
    for l in backbone.layers[-15:]:
        print(f'  {l.name:<40} {type(l).__name__}')


In [ ]:
def build_gradcam_model(full_model, backbone, last_conv_name):
    """Build a model that outputs last conv activations AND final predictions."""
    feature_extractor = keras.Model(
        inputs=backbone.inputs,
        outputs=backbone.get_layer(last_conv_name).output,
        name='gradcam_feature_extractor',
    )
    img_input = keras.Input(shape=INPUT_SHAPE, name='gradcam_input')
    features  = feature_extractor(img_input)
    preds     = full_model(img_input)
    return keras.Model(inputs=img_input, outputs=[features, preds], name='gradcam_model')


gradcam_model = build_gradcam_model(best_model, backbone, LAST_CONV_NAME)
print(f'Grad-CAM model built. Target layer: {LAST_CONV_NAME}')


In [ ]:
def make_gradcam_heatmap(grad_model, img_array):
    """Generate Grad-CAM heatmap for the predicted class."""
    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array, training=False)
        pred_class  = tf.argmax(predictions[0]).numpy()
        confidence  = float(predictions[0, pred_class])
        class_score = predictions[:, pred_class]
    grads = tape.gradient(class_score, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_outputs = conv_outputs[0]
    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap).numpy()
    heatmap = np.maximum(heatmap, 0)
    heatmap = heatmap / (np.max(heatmap) + 1e-8)
    return heatmap, pred_class, confidence


def display_gradcam_grid(gradcam_model, dataset, class_names, output_dir,
                          n_samples=4, model_name='MobileNetV3Large'):
    images_batch, labels_batch = next(iter(dataset))
    n = min(n_samples, len(images_batch))
    fig, axes = plt.subplots(n, 3, figsize=(12, n * 3.8))
    if n == 1:
        axes = [axes]

    for i in range(n):
        img       = images_batch[i:i+1]
        true_cls  = class_names[np.argmax(labels_batch[i].numpy())]
        heatmap, pred_class, confidence = make_gradcam_heatmap(gradcam_model, img)
        pred_cls  = class_names[pred_class]

        img_display      = np.clip((img[0].numpy() + 1.0) / 2.0, 0, 1)
        heatmap_resized  = np.array(
            tf.image.resize(heatmap[..., np.newaxis], IMAGE_SIZE)
        ).squeeze()
        heatmap_rgb      = cm.jet(heatmap_resized)[..., :3]
        overlay          = np.clip(0.55 * img_display + 0.45 * heatmap_rgb, 0, 1)
        correct_marker   = '\u2713' if pred_class == np.argmax(labels_batch[i].numpy()) else '\u2717'

        axes[i][0].imshow(img_display)
        axes[i][0].set_title(f'Original\nTrue: {true_cls}', fontsize=10)
        axes[i][0].axis('off')
        axes[i][1].imshow(heatmap_resized, cmap='jet')
        axes[i][1].set_title('Grad-CAM Heatmap', fontsize=10)
        axes[i][1].axis('off')
        axes[i][2].imshow(overlay)
        axes[i][2].set_title(f'Overlay {correct_marker}\nPred: {pred_cls} ({confidence:.1%})', fontsize=10)
        axes[i][2].axis('off')

    plt.suptitle(f'{model_name} — Grad-CAM Explanations', fontsize=14)
    plt.tight_layout()
    fname = os.path.join(output_dir, 'gradcam_visualizations.png')
    plt.savefig(fname, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Grad-CAM saved: {fname}')


display_gradcam_grid(gradcam_model, test_ds, CLASS_NAMES, OUTPUT_DIR, n_samples=4)


## Section 14: Model Calibration (ECE)

In [ ]:
def compute_ece_and_plot(true_labels, pred_probs, predictions,
                          n_bins=10, output_dir=None, model_name='Model'):
    confidences = np.max(pred_probs, axis=1)
    correct     = (predictions == true_labels).astype(float)
    N           = len(true_labels)
    bin_edges   = np.linspace(0, 1, n_bins + 1)
    bin_acc, bin_conf, bin_cnt = [], [], []
    for lo, hi in zip(bin_edges[:-1], bin_edges[1:]):
        mask = (confidences >= lo) & (confidences < hi)
        if mask.sum() > 0:
            bin_acc.append(correct[mask].mean())
            bin_conf.append(confidences[mask].mean())
            bin_cnt.append(mask.sum())
        else:
            bin_acc.append(0.0); bin_conf.append((lo+hi)/2); bin_cnt.append(0)
    ece = sum(cnt/N * abs(a - c) for cnt, a, c in zip(bin_cnt, bin_acc, bin_conf))

    fig, ax = plt.subplots(figsize=(7, 7))
    width = 1.0 / n_bins * 0.85
    ax.bar(bin_conf, bin_acc, width=width, alpha=0.75, color='steelblue',
           label='Accuracy per bin', zorder=3)
    ax.bar(bin_conf, [c - a for c, a in zip(bin_conf, bin_acc)],
           bottom=bin_acc, width=width, alpha=0.3, color='red',
           label='Gap', zorder=3)
    ax.plot([0, 1], [0, 1], 'k--', lw=2, label='Perfect calibration')
    ax.set(xlabel='Confidence', ylabel='Accuracy', xlim=[0, 1], ylim=[0, 1],
           title=f'{model_name} — Reliability Diagram\nECE = {ece:.4f}')
    ax.legend(fontsize=10); ax.grid(True, alpha=0.3, zorder=0)
    plt.tight_layout()
    if output_dir:
        fname = os.path.join(output_dir, 'calibration_reliability_diagram.png')
        plt.savefig(fname, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Expected Calibration Error (ECE): {ece:.4f}')
    return ece


ece = compute_ece_and_plot(all_true, all_probs, all_preds, output_dir=OUTPUT_DIR,
                            model_name='MobileNetV3Large')


## Section 15: OOD Rejection Analysis

In [ ]:
confidences  = np.max(all_probs, axis=1)
correct_mask = (all_preds == all_true)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
axes[0].hist(confidences[correct_mask],  bins=25, alpha=0.7, color='green',
             density=True, label=f'Correct  ({correct_mask.sum()})')
axes[0].hist(confidences[~correct_mask], bins=25, alpha=0.7, color='red',
             density=True, label=f'Incorrect ({(~correct_mask).sum()})')
axes[0].axvline(CONFIDENCE_THRESHOLD, color='black', ls='--', lw=2,
                label=f'Threshold = {CONFIDENCE_THRESHOLD}')
axes[0].set(xlabel='Confidence (max softmax)', ylabel='Density',
             title='Confidence Distribution on Test Set')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

thresholds = np.arange(0.05, 1.0, 0.05)
coverages, accs_at_thresh = [], []
for t in thresholds:
    mask = confidences >= t
    coverages.append(mask.mean())
    accs_at_thresh.append(
        accuracy_score(all_true[mask], all_preds[mask]) if mask.sum() > 0 else 1.0
    )

axes[1].plot(coverages, accs_at_thresh, 'b-o', markersize=5, lw=2)
axes[1].axvline((confidences >= CONFIDENCE_THRESHOLD).mean(),
                color='red', ls='--', lw=1.5,
                label=f'Threshold={CONFIDENCE_THRESHOLD}')
axes[1].set(xlabel='Coverage', ylabel='Accuracy on Accepted',
             title='Accuracy-Coverage Trade-off')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.suptitle('MobileNetV3Large — OOD Rejection Analysis', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'ood_analysis.png'), dpi=150, bbox_inches='tight')
plt.show()

accepted = confidences >= CONFIDENCE_THRESHOLD
print(f'OOD Threshold = {CONFIDENCE_THRESHOLD}')
print(f'  Accepted: {accepted.sum()}/{len(confidences)} ({accepted.mean():.1%})')
print(f'  Rejected: {(~accepted).sum()}/{len(confidences)} ({(~accepted).mean():.1%})')
if accepted.sum() > 0:
    print(f'  Accuracy on accepted: {accuracy_score(all_true[accepted], all_preds[accepted]):.4f}')

epsilon = 1e-9
entropy = -np.sum(all_probs * np.log(all_probs + epsilon), axis=1)
norm_entropy = entropy / np.log(NUM_CLASSES)
print(f'\nEntropy (normalized, 0=certain, 1=max uncertainty):')
print(f'  All — mean: {norm_entropy.mean():.4f}  std: {norm_entropy.std():.4f}')
print(f'  Correct   — mean: {norm_entropy[correct_mask].mean():.4f}')
print(f'  Incorrect — mean: {norm_entropy[~correct_mask].mean():.4f}')


## Section 16: TFLite Conversion

In [ ]:
best_model.save(SAVEDMODEL_PATH)
print(f'SavedModel saved: {SAVEDMODEL_PATH}')


In [ ]:
def convert_tflite(savedmodel_path, output_path, quantization=None,
                   representative_data_fn=None):
    converter = tf.lite.TFLiteConverter.from_saved_model(savedmodel_path)
    if quantization == 'float16':
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        converter.target_spec.supported_types = [tf.float16]
    elif quantization == 'int8':
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        if representative_data_fn:
            converter.representative_dataset = representative_data_fn
            converter.inference_input_type  = tf.float32
            converter.inference_output_type = tf.float32
    tflite_model = converter.convert()
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    with open(output_path, 'wb') as f:
        f.write(tflite_model)
    size_kb = os.path.getsize(output_path) / 1024
    print(f'  \u2705 {os.path.basename(output_path)}: {size_kb:.1f} KB')
    return size_kb


def representative_dataset_gen():
    count = 0
    for images, _ in train_ds:
        for img in images:
            if count >= 100:
                return
            yield [tf.expand_dims(img, 0)]
            count += 1


print('Converting to TFLite...')
size_f32  = convert_tflite(SAVEDMODEL_PATH, TFLITE_PATH_F32,  quantization=None)
size_f16  = convert_tflite(SAVEDMODEL_PATH, TFLITE_PATH_F16,  quantization='float16')
size_int8 = convert_tflite(SAVEDMODEL_PATH, TFLITE_PATH_INT8, quantization='int8',
                            representative_data_fn=representative_dataset_gen)

print(f'\nSize summary:')
print(f'  Float32: {size_f32:.1f} KB  (baseline)')
print(f'  Float16: {size_f16:.1f} KB  ({size_f32/size_f16:.1f}x smaller)')
print(f'  Int8:    {size_int8:.1f} KB  ({size_f32/size_int8:.1f}x smaller)')


In [ ]:
def verify_tflite(tflite_path, test_dataset, class_names, label):
    interpreter = tf.lite.Interpreter(model_path=tflite_path)
    interpreter.allocate_tensors()
    inp_det = interpreter.get_input_details()[0]
    out_det = interpreter.get_output_details()[0]
    all_tflite_preds, all_tflite_true = [], []
    for images, labels in test_dataset:
        for img, lbl in zip(images.numpy(), labels.numpy()):
            inp = np.expand_dims(img.astype(inp_det['dtype']), 0)
            interpreter.set_tensor(inp_det['index'], inp)
            interpreter.invoke()
            output = interpreter.get_tensor(out_det['index'])[0]
            all_tflite_preds.append(np.argmax(output))
            all_tflite_true.append(np.argmax(lbl))
    tflite_acc = accuracy_score(all_tflite_true, all_tflite_preds)
    print(f'  {label}: dtype={inp_det["dtype"].__name__}, '
          f'shape={inp_det["shape"].tolist()}, accuracy={tflite_acc:.4f}')
    return tflite_acc


print('\nVerifying TFLite models on test set...')
acc_f32  = verify_tflite(TFLITE_PATH_F32,  test_ds, CLASS_NAMES, 'Float32')
acc_f16  = verify_tflite(TFLITE_PATH_F16,  test_ds, CLASS_NAMES, 'Float16')
acc_int8 = verify_tflite(TFLITE_PATH_INT8, test_ds, CLASS_NAMES, 'Int8')

print(f'\nAccuracy drop from Float32 baseline:')
print(f'  Float16 drop: {(acc_f32 - acc_f16)*100:+.2f}%')
print(f'  Int8 drop:    {(acc_f32 - acc_int8)*100:+.2f}%')


## Section 17: MobileNetV2 vs MobileNetV3Large Comparison

After running both notebooks, populate this table with your actual results.

**Important:** This comparison is only valid if both models were trained and evaluated on the **same dataset split** using the **same random seed**.

In [ ]:
# ============================================================
# COMPARISON TABLE
# Fill in MobileNetV2 values from the MobileNetV2 notebook output.
# MobileNetV3Large values come from this notebook's results.
# ============================================================

# Placeholders — replace with your actual MobileNetV2 results
V2_ACC   = None  # e.g. 0.9250
V2_PREC  = None
V2_REC   = None
V2_F1    = None
V2_AUC   = None
V2_ECE   = None
V2_TTA   = None
V2_F32   = None  # TFLite float32 size in KB
V2_F16   = None
V2_INT8  = None

comparison_data = {
    'Metric': [
        'Test Accuracy',
        'TTA Accuracy',
        'Precision (weighted)',
        'Recall (weighted)',
        'F1 Score (weighted)',
        'Macro AUC',
        'Calibration (ECE)',
        'TFLite Float32 (KB)',
        'TFLite Float16 (KB)',
        'TFLite Int8 (KB)',
    ],
    'MobileNetV2': [
        V2_ACC,  V2_TTA,  V2_PREC, V2_REC,  V2_F1,
        V2_AUC,  V2_ECE,  V2_F32,  V2_F16,  V2_INT8,
    ],
    'MobileNetV3Large': [
        acc,      tta_acc, prec,    rec,     f1,
        macro_auc, ece,   size_f32, size_f16, size_int8,
    ],
}

comparison_df = pd.DataFrame(comparison_data)

# Format for display
def fmt(val):
    if val is None:
        return 'N/A (run MobileNetV2 notebook)'
    if isinstance(val, float) and val < 10:
        return f'{val:.4f}'
    return f'{val:.1f}'

comparison_df['MobileNetV2']      = comparison_df['MobileNetV2'].apply(fmt)
comparison_df['MobileNetV3Large'] = comparison_df['MobileNetV3Large'].apply(fmt)

print('\n' + '=' * 70)
print('MODEL COMPARISON: MobileNetV2 vs MobileNetV3Large')
print('Dataset: PechayDetect  |  Seed: 42  |  Input: 224x224x3')
print('=' * 70)
print(comparison_df.to_string(index=False))
print('=' * 70)
print('Note: Lower ECE = better calibration. Lower KB = smaller model.')

comparison_df.to_csv(os.path.join(OUTPUT_DIR, 'model_comparison.csv'), index=False)
print(f'\nComparison table saved: {OUTPUT_DIR}/model_comparison.csv')


## Section 18: Final Summary Report

In [ ]:
print('\n' + '=' * 65)
print('MOBILENETV3LARGE — FINAL EVALUATION REPORT')
print('=' * 65)
print(f'  Architecture:       MobileNetV3Large (alpha=1.0, minimalistic=False)')
print(f'  Input size:         {IMAGE_SIZE[0]}x{IMAGE_SIZE[1]}x{CHANNELS}')
print(f'  Classes ({NUM_CLASSES}):       {CLASS_NAMES}')
print(f'  Training phases:    3 (Head | Top Fine-Tune | Full Fine-Tune)')
print(f'  Augmentation:       Flip, Rotation\u00b115\u00b0, Zoom\u00b110%, Brightness\u00b120%, Contrast\u00b110%')
print(f'  Label smoothing:    {LABEL_SMOOTHING}')
print(f'  Dropout:            {DROPOUT_RATE}')
print(f'  L2 regularization:  {L2_REGULARIZATION}')
print(f'  OOD threshold:      {CONFIDENCE_THRESHOLD}')
print('-' * 65)
print('  TEST SET PERFORMANCE')
print(f'    Accuracy  (standard): {acc:.4f}  ({acc*100:.2f}%)')
print(f'    Accuracy  (TTA):      {tta_acc:.4f}  ({tta_acc*100:.2f}%)')
print(f'    Precision (weighted): {prec:.4f}')
print(f'    Recall    (weighted): {rec:.4f}')
print(f'    F1 Score  (weighted): {f1:.4f}')
print(f'    Macro AUC:            {macro_auc:.4f}')
print(f'    Calibration (ECE):    {ece:.4f}')
print('-' * 65)
print('  TFLITE MODELS')
print(f'    Float32:  {size_f32:.1f} KB   accuracy: {acc_f32:.4f}')
print(f'    Float16:  {size_f16:.1f} KB   accuracy: {acc_f16:.4f}  (recommended for Android)')
print(f'    Int8:     {size_int8:.1f} KB   accuracy: {acc_int8:.4f}')
print('-' * 65)
print('  OUTPUT FILES')
for f in sorted(Path(OUTPUT_DIR).rglob('*')):
    if f.is_file():
        size = f.stat().st_size / 1024
        print(f'    {f.name:<45} {size:>8.1f} KB')
print('=' * 65)
print('\u2705 MobileNetV3Large notebook complete.')
print('Fill in MobileNetV2 values in Section 17 to generate the comparison table.')
